<a href="https://colab.research.google.com/github/lipikamallikarjuna/MissingRfidDetection-ANSC6060/blob/main/TrajectoryMatching6060.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, pickle, os
from scipy.optimize import curve_fit
from itertools import product
from tqdm import tqdm

ProjectRoot = Path("/content/drive/MyDrive/MiniProject")
print("Ready")

Ready


In [ ]:
DfKnown   = pd.read_csv(ProjectRoot / "DfKnown_clean.csv",
                         parse_dates=["EventDate"], low_memory=False)
DfMissing = pd.read_csv(ProjectRoot / "DfMissing_clean.csv",
                         parse_dates=["EventDate"], low_memory=False)

DfKnown["AnimalId"]   = DfKnown["AnimalId"].astype(str)
DfMissing["AnimalId"] = DfMissing["AnimalId"].astype(str)

DfGaps = pd.read_csv(ProjectRoot / "DfGaps.csv",
                      parse_dates=["PreDate","GapStart","GapEnd",
                                   "PostDate","CalvingDate"])

with open(ProjectRoot / "AllWoodsParams.pkl","rb") as f:
    AllWoodsParams = pickle.load(f)
with open(ProjectRoot / "AnimalsByDate.pkl","rb") as f:
    AnimalsByDate = pickle.load(f)

Features = ["TotalMilkYieldSessionKg","MilkingDurationSeconds",
            "AverageMilkFlowKgPerMin","MilkFlow30To60SecondsKgPerMin",
            "MilkYieldFirst2MinutesKg"]

def WoodsCurve(T, A, B, C):
    return A * (T**B) * np.exp(-C * T)

print(f"DfKnown: {len(DfKnown):,}  DfMissing: {len(DfMissing):,}")
print(f"Wood's params: {len(AllWoodsParams):,} animals")
print(f"AnimalsByDate: {len(AnimalsByDate):,} dates")
print("READY")

DfKnown: 6,737,364  DfMissing: 1,701,003
Wood's params: 5,610 animals
AnimalsByDate: 861 dates
READY


In [ ]:
# Pre-index mystery records by date+session for fast lookup
MysteryByDateSession = {}
for (date, session), grp in DfMissing.groupby(
        [DfMissing["EventDate"].dt.date, "MilkingSession"]):
    MysteryByDateSession[(date, session)] = grp[Features + ["EventDate","MilkingSession"]].copy().reset_index(drop=True)

print(f"Mystery date-sessions indexed: {len(MysteryByDateSession):,}")

Mystery date-sessions indexed: 2,490


In [ ]:
WINDOW       = 7
YIELD_STD    = 1.1
MAX_MEAN_ERR = 0.45
FILTER_STD   = 1.5

def get_woods_pred(aid, lac, dim):
    """Try (aid, lac) tuple first, then aid-only key."""
    aid = str(aid)
    # try tuple key first (some versions saved this way)
    for key in [(aid, int(lac)), (aid, float(lac))]:
        if key in AllWoodsParams:
            A, B, C = AllWoodsParams[key]
            try:
                pred = A * (dim**B) * np.exp(-C * dim)
                return float(pred) if pred > 0 else None
            except:
                return None
    # try aid-only key
    if aid in AllWoodsParams:
        params = AllWoodsParams[aid]
        # might be dict of lacs, or direct (A,B,C)
        if isinstance(params, dict):
            lac_key = int(lac)
            if lac_key not in params:
                lac_key = float(lac)
            if lac_key not in params:
                return None
            A, B, C = params[lac_key]
        else:
            A, B, C = params
        try:
            pred = A * (dim**B) * np.exp(-C * dim)
            return float(pred) if pred > 0 else None
        except:
            return None
    return None

def score_trajectory(animal_id, gap_row, dates_window, session):
    animal_id  = str(animal_id)
    calving    = gap_row["CalvingDate"]
    pre_lac    = int(gap_row["PreLac"])
    post_lac   = int(gap_row["PostLac"])
    pre_dim    = int(gap_row["PreDIM"])
    gap_start  = pd.Timestamp(gap_row["GapStart"]).date()

    day_candidates = []
    for d in dates_window:
        d_ts = pd.Timestamp(d)
        # determine lactation and DIM
        if not pd.isnull(calving) and d_ts >= calving:
            lac = post_lac
            dim = (d_ts - calving).days
        else:
            lac = pre_lac
            days_in = (d_ts - pd.Timestamp(gap_start)).days
            dim = pre_dim + days_in

        if dim <= 0:
            return None

        pred = get_woods_pred(animal_id, lac, dim)
        if pred is None or pred < 3.0:
            return None

        key = (d, session)
        if key not in MysteryByDateSession or len(MysteryByDateSession[key]) == 0:
            return None

        pool = MysteryByDateSession[key].copy()
        pool["_pred"] = pred
        pool["_err"]  = (pool["TotalMilkYieldSessionKg"] - pred).abs()
        candidates = pool[pool["_err"] <= FILTER_STD * YIELD_STD].copy()
        if len(candidates) == 0:
            return None
        day_candidates.append(candidates)

    if len(day_candidates) < WINDOW:
        return None

    errors = []
    idxs   = []
    for dc in day_candidates:
        best_row = dc.loc[dc["_err"].idxmin()]
        errors.append(best_row["_err"])
        idxs.append(int(best_row.name))

    mean_err = float(np.mean(errors))
    if mean_err <= MAX_MEAN_ERR:
        return mean_err, idxs, [float(dc["_pred"].iloc[0]) for dc in day_candidates]
    return None

# quick sanity check on the test animal
import datetime
test_row  = DfGaps.iloc[0]
test_aid  = str(test_row["AnimalId"])
gap_start = test_row["GapStart"].date()
window    = [gap_start + datetime.timedelta(days=i) for i in range(WINDOW)]

for s in [1, 3]:   # skip session 2 — mostly empty
    res = score_trajectory(test_aid, test_row, window, s)
    print(f"Session {s}: {res}")

Session 1: (0.0154107322024295, [36, 30, 121, 54, 126, 102, 51], [12.205140625057204, 12.184464792056762, 12.163786340287071, 12.143105659914147, 12.122423136505573, 12.1017391510873, 12.081054080199511])
Session 3: (0.011076790998735677, [157, 293, 55, 323, 57, 103, 579], [12.205140625057204, 12.184464792056762, 12.163786340287071, 12.143105659914147, 12.122423136505573, 12.1017391510873, 12.081054080199511])


In [ ]:
import datetime
np.random.seed(42)

animal_dates = (DfKnown.groupby("AnimalId")["EventDate"]
                .apply(lambda x: sorted(x.dt.date.unique()))
                .reset_index())
animal_dates.columns = ["AnimalId","Dates"]
animal_dates["NDates"] = animal_dates["Dates"].apply(len)

candidates_val = animal_dates[
    animal_dates["NDates"] >= WINDOW + 10
]["AnimalId"].values

sample_animals = np.random.choice(
    candidates_val, min(300, len(candidates_val)), replace=False
)
print(f"Validation pool: {len(sample_animals)} animals")

results = []

for aid in tqdm(sample_animals, desc="Validating"):
    aid = str(aid)
    animal_recs = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 6:
        continue

    # pick session with most records (avoid empty sessions)
    session_counts = animal_recs["MilkingSession"].value_counts()
    session = int(session_counts.idxmax())

    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    # held-out records for this animal+session in the window
    mask = (animal_recs["EventDate"].dt.date.isin(window_dates) &
            (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    # get or build gap_row
    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_rec  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]].iloc[-1]
        post_rec = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]].iloc[0]
        gap_row  = pd.Series({
            "AnimalId":    aid,
            "GapStart":    pd.Timestamp(window_dates[0]),
            "GapEnd":      pd.Timestamp(window_dates[-1]),
            "PreDIM":      pre_rec.get("DaysInMilk", 150),
            "PreLac":      pre_rec.get("LactationNumber", 1),
            "PostLac":     post_rec.get("LactationNumber", 1),
            "CalvingDate": pd.NaT,
        })

    # inject held-out records into mystery pool temporarily
    injected_keys = []
    for d in window_dates:
        key = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0:
            continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(
                columns=Features + ["EventDate","MilkingSession"])
        MysteryByDateSession[key] = pd.concat(
            [MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)

    # run scorer
    result = score_trajectory(aid, gap_row, window_dates, session)

    # clean up injected records
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    recovered = result is not None
    results.append({
        "AnimalId":  aid,
        "recovered": recovered,
        "mean_err":  result[0] if recovered else None,
    })

df_val = pd.DataFrame(results)
n_rec   = df_val["recovered"].sum()
n_total = len(df_val)
print(f"\nValidation results:")
print(f"  Animals tested        : {n_total}")
print(f"  Recovered             : {n_rec}  ({100*n_rec/n_total:.1f}%)")
if n_rec > 0:
    print(f"  Mean error (recovered): {df_val[df_val['recovered']]['mean_err'].mean():.4f} kg")
    print(f"  Error distribution:")
    print(df_val[df_val['recovered']]['mean_err'].describe())

Validation pool: 300 animals


Validating:  53%|█████▎    | 160/300 [01:55<01:41,  1.38it/s]


KeyboardInterrupt: 

In [ ]:
import datetime
np.random.seed(99)

# Stricter test: inject MULTIPLE animals simultaneously,
# check if each one recovers itself and not someone else's record

sample2 = np.random.choice(candidates_val, min(100, len(candidates_val)), replace=False)

identity_results = []

for aid in tqdm(sample2, desc="Identity check"):
    aid = str(aid)
    animal_recs = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 6:
        continue

    session_counts = animal_recs["MilkingSession"].value_counts()
    session = int(session_counts.idxmax())

    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    mask = (animal_recs["EventDate"].dt.date.isin(window_dates) &
            (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_rec  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]].iloc[-1]
        post_rec = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]].iloc[0]
        gap_row  = pd.Series({
            "AnimalId":    aid,
            "GapStart":    pd.Timestamp(window_dates[0]),
            "GapEnd":      pd.Timestamp(window_dates[-1]),
            "PreDIM":      pre_rec.get("DaysInMilk", 150),
            "PreLac":      pre_rec.get("LactationNumber", 1),
            "PostLac":     post_rec.get("LactationNumber", 1),
            "CalvingDate": pd.NaT,
        })

    # inject held-out records — tag each injected row so we can identify it
    injected_keys = []
    injected_indices = {}  # date -> original index in held_out

    for d in window_dates:
        key = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0:
            continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(
                columns=Features + ["EventDate","MilkingSession"])
        new_idx = len(MysteryByDateSession[key])
        MysteryByDateSession[key] = pd.concat(
            [MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)
        injected_indices[d] = new_idx  # we know the correct record is at this index

    result = score_trajectory(aid, gap_row, window_dates, session)

    # check identity: did the recovered indices match the injected ones?
    correct_days = 0
    if result is not None:
        mean_err, found_idxs, preds = result
        for i, d in enumerate(window_dates):
            if d in injected_indices:
                if found_idxs[i] == injected_indices[d]:
                    correct_days += 1

    # clean up
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    identity_results.append({
        "AnimalId":     aid,
        "recovered":    result is not None,
        "mean_err":     result[0] if result is not None else None,
        "correct_days": correct_days,
        "window_days":  WINDOW,
        "pct_correct":  correct_days / WINDOW if result is not None else 0,
    })

df_id = pd.DataFrame(identity_results)
n_rec  = df_id["recovered"].sum()
n_tot  = len(df_id)
rec_df = df_id[df_id["recovered"]]

print(f"Identity accuracy check ({n_tot} animals):")
print(f"  Recovered            : {n_rec} ({100*n_rec/n_tot:.1f}%)")
if n_rec > 0:
    print(f"  Mean error           : {rec_df['mean_err'].mean():.4f} kg")
    print(f"  Days correct / window: {rec_df['correct_days'].mean():.2f} / {WINDOW}")
    print(f"  % days correct       : {100*rec_df['pct_correct'].mean():.1f}%")
    print(f"  All 7 days correct   : {(rec_df['correct_days']==WINDOW).sum()} animals ({100*(rec_df['correct_days']==WINDOW).mean():.1f}%)")

Identity check: 100%|██████████| 100/100 [01:14<00:00,  1.34it/s]

Identity accuracy check (40 animals):
  Recovered            : 16 (40.0%)
  Mean error           : 0.0971 kg
  Days correct / window: 1.00 / 7
  % days correct       : 14.3%
  All 7 days correct   : 0 animals (0.0%)


In [ ]:
from scipy.optimize import linear_sum_assignment

WINDOW       = 7
MAX_MEAN_ERR = 0.45
FILTER_STD   = 1.5

def score_trajectory_hungarian(animal_ids, gap_rows_dict, dates_window, session):
    """
    Run Hungarian across all gap animals simultaneously for the full window.
    For each animal × each day, get Wood's predicted yield.
    Build one big cost matrix: rows = animals, cols = mystery records (flattened across days).

    Actually: run per-day Hungarian (as before) but use cross-day consistency
    as a secondary score — does the same animal get assigned records that
    follow their predicted trajectory shape across days?

    Returns dict: animal_id -> (mean_err, [record_idxs per day])
    """
    # Step 1: for each day, get predictions for all animals
    # shape: animals × days
    day_pools    = {}   # date -> mystery pool df
    day_preds    = {}   # (aid, date) -> predicted yield

    for d in dates_window:
        key = (d, session)
        if key not in MysteryByDateSession or len(MysteryByDateSession[key]) == 0:
            return {}
        day_pools[d] = MysteryByDateSession[key].copy().reset_index(drop=True)

    # get predictions per animal per day
    valid_animals = []
    for aid in animal_ids:
        gap_row   = gap_rows_dict[aid]
        calving   = gap_row["CalvingDate"]
        pre_lac   = int(gap_row["PreLac"])
        post_lac  = int(gap_row["PostLac"])
        pre_dim   = int(gap_row["PreDIM"])
        gap_start = pd.Timestamp(gap_row["GapStart"]).date()

        preds = {}
        valid = True
        for d in dates_window:
            d_ts = pd.Timestamp(d)
            if not pd.isnull(calving) and d_ts >= calving:
                lac = post_lac
                dim = (d_ts - calving).days
            else:
                lac = pre_lac
                dim = pre_dim + (d_ts - pd.Timestamp(gap_start)).days
            if dim <= 0:
                valid = False; break
            pred = get_woods_pred(aid, lac, dim)
            if pred is None or pred < 3.0:
                valid = False; break
            preds[d] = pred

        if valid:
            valid_animals.append(aid)
            for d, p in preds.items():
                day_preds[(aid, d)] = p

    if len(valid_animals) == 0:
        return {}

    # Step 2: per day, run Hungarian to get best assignment
    # Then score cross-day consistency
    day_assignments = {}  # date -> {aid: record_idx}

    for d in dates_window:
        pool   = day_pools[d]
        n_rec  = len(pool)
        n_ani  = len(valid_animals)

        # cost matrix: animals × records
        cost = np.full((n_ani, n_rec), 999.0)
        for i, aid in enumerate(valid_animals):
            pred = day_preds.get((aid, d))
            if pred is None:
                continue
            errs = (pool["TotalMilkYieldSessionKg"] - pred).abs()
            # only consider records within filter range
            mask = errs <= FILTER_STD * YIELD_STD
            cost[i, mask] = (errs[mask] / YIELD_STD).values

        row_ind, col_ind = linear_sum_assignment(cost)
        day_assignments[d] = {}
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < 999.0:
                day_assignments[d][valid_animals[r]] = (c, cost[r,c])

    # Step 3: score each animal across days
    results = {}
    for aid in valid_animals:
        day_errors = []
        day_idxs   = []
        for d in dates_window:
            if aid not in day_assignments.get(d, {}):
                break
            idx, err = day_assignments[d][aid]
            day_errors.append(err * YIELD_STD)  # back to kg
            day_idxs.append(idx)

        if len(day_errors) == WINDOW:
            mean_err = np.mean(day_errors)
            if mean_err <= MAX_MEAN_ERR:
                results[aid] = (mean_err, day_idxs)

    return results

print("Multi-day Hungarian scorer ready")

Multi-day Hungarian scorer ready


In [ ]:
np.random.seed(42)
sample_A = np.random.choice(candidates_val, min(100, len(candidates_val)), replace=False)

results_A = []

for aid in tqdm(sample_A, desc="Option A validation"):
    aid = str(aid)
    animal_recs  = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 6:
        continue

    session_counts = animal_recs["MilkingSession"].value_counts()
    session = int(session_counts.idxmax())

    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    mask = (animal_recs["EventDate"].dt.date.isin(window_dates) &
            (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    # get gap_row
    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_rec  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]].iloc[-1]
        post_rec = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]].iloc[0]
        gap_row  = pd.Series({
            "AnimalId":    aid,
            "GapStart":    pd.Timestamp(window_dates[0]),
            "GapEnd":      pd.Timestamp(window_dates[-1]),
            "PreDIM":      pre_rec.get("DaysInMilk", 150),
            "PreLac":      pre_rec.get("LactationNumber", 1),
            "PostLac":     post_rec.get("LactationNumber", 1),
            "CalvingDate": pd.NaT,
        })

    # build gap_rows_dict with ALL gap animals on these dates
    # so Hungarian competes across all of them simultaneously
    date_animals = set()
    for d in window_dates:
        date_animals.update(
            [str(x) for x in AnimalsByDate.get(d, [])]
        )
    date_animals.add(aid)  # make sure target is included

    gap_rows_dict = {}
    for a in date_animals:
        gr = DfGaps[DfGaps["AnimalId"].astype(str) == a]
        if len(gr) > 0:
            gap_rows_dict[a] = gr.iloc[0]

    if aid not in gap_rows_dict:
        gap_rows_dict[aid] = gap_row

    # inject held-out records
    injected_keys    = []
    injected_indices = {}
    for d in window_dates:
        key     = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0:
            continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(
                columns=Features + ["EventDate","MilkingSession"])
        new_idx = len(MysteryByDateSession[key])
        MysteryByDateSession[key] = pd.concat(
            [MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)
        injected_indices[d] = new_idx

    # run multi-day Hungarian
    res = score_trajectory_hungarian(
        list(gap_rows_dict.keys()), gap_rows_dict, window_dates, session
    )

    # check identity
    correct_days = 0
    recovered    = aid in res
    if recovered:
        mean_err, found_idxs = res[aid]
        for i, d in enumerate(window_dates):
            if d in injected_indices and found_idxs[i] == injected_indices[d]:
                correct_days += 1

    # clean up
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    results_A.append({
        "AnimalId":     aid,
        "recovered":    recovered,
        "mean_err":     res[aid][0] if recovered else None,
        "correct_days": correct_days,
        "pct_correct":  correct_days / WINDOW if recovered else 0,
    })

df_A = pd.DataFrame(results_A)
n_rec = df_A["recovered"].sum()
n_tot = len(df_A)
rec   = df_A[df_A["recovered"]]

print(f"\nOption A — Multi-day Hungarian ({n_tot} animals):")
print(f"  Recovered            : {n_rec} ({100*n_rec/n_tot:.1f}%)")
if n_rec > 0:
    print(f"  Mean error           : {rec['mean_err'].mean():.4f} kg")
    print(f"  Days correct / window: {rec['correct_days'].mean():.2f} / {WINDOW}")
    print(f"  % days correct       : {100*rec['pct_correct'].mean():.1f}%")
    print(f"  All 7 days correct   : {(rec['correct_days']==WINDOW).sum()} ({100*(rec['correct_days']==WINDOW).mean():.1f}%)")

Option A validation: 100%|██████████| 100/100 [01:10<00:00,  1.42it/s]


Option A — Multi-day Hungarian (32 animals):
  Recovered            : 0 (0.0%)


In [ ]:
from scipy.optimize import linear_sum_assignment

np.random.seed(42)
sample_val6 = np.random.choice(candidates_val, min(200, len(candidates_val)), replace=False)

val6_results = []

for aid in tqdm(sample_val6, desc="Full pipeline validation"):
    aid = str(aid)
    animal_recs  = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 10:
        continue

    # pick session with most records
    session = int(animal_recs["MilkingSession"].value_counts().idxmax())

    # pick a random 7-day window to hold out
    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    # held-out records = the "mystery" records for this animal
    mask     = (animal_recs["EventDate"].dt.date.isin(window_dates) &
                (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    # ── STEP 1: build gap_row from known info ──
    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_recs  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]]
        post_recs = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]]
        if len(pre_recs) == 0 or len(post_recs) == 0:
            continue
        pre_rec  = pre_recs.iloc[-1]
        post_rec = post_recs.iloc[0]
        gap_row  = pd.Series({
            "AnimalId":    aid,
            "GapStart":    pd.Timestamp(window_dates[0]),
            "GapEnd":      pd.Timestamp(window_dates[-1]),
            "PreDIM":      pre_rec.get("DaysInMilk", 150),
            "PreLac":      pre_rec.get("LactationNumber", 1),
            "PostLac":     post_rec.get("LactationNumber", 1),
            "CalvingDate": pd.NaT,
        })

    calving  = gap_row["CalvingDate"]
    pre_lac  = int(gap_row["PreLac"])
    post_lac = int(gap_row["PostLac"])
    pre_dim  = int(gap_row["PreDIM"])
    gap_start = pd.Timestamp(gap_row["GapStart"]).date()

    # ── STEP 2: get Wood's predictions for each window day ──
    day_preds = {}
    valid = True
    for d in window_dates:
        d_ts = pd.Timestamp(d)
        if not pd.isnull(calving) and d_ts >= calving:
            lac = post_lac
            dim = (d_ts - calving).days
        else:
            lac = pre_lac
            dim = pre_dim + (d_ts - pd.Timestamp(gap_start)).days
        if dim <= 0:
            valid = False; break
        pred = get_woods_pred(aid, lac, dim)
        if pred is None or pred < 3.0:
            valid = False; break
        day_preds[d] = pred
    if not valid:
        continue

    # ── inject held-out records into mystery pool ──
    injected_keys    = []
    injected_indices = {}
    for d in window_dates:
        key     = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0:
            continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(
                columns=Features + ["EventDate","MilkingSession"])
        new_idx = len(MysteryByDateSession[key])
        MysteryByDateSession[key] = pd.concat(
            [MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)
        injected_indices[d] = new_idx

    # ── STEP 3+4: pre-filter + Hungarian per day ──
    day_results  = {}
    all_days_ok  = True

    for d in window_dates:
        pred = day_preds[d]
        key  = (d, session)
        if key not in MysteryByDateSession or len(MysteryByDateSession[key]) == 0:
            all_days_ok = False; break

        pool = MysteryByDateSession[key].copy().reset_index(drop=True)

        # get all gap animals absent on this date for competition
        absent_aids = [str(x) for x in AnimalsByDate.get(d, [])]
        if aid not in absent_aids:
            absent_aids.append(aid)

        # get predictions for all competing animals
        competitor_preds = {}
        for comp_aid in absent_aids:
            comp_gaps = DfGaps[DfGaps["AnimalId"].astype(str) == comp_aid]
            if len(comp_gaps) == 0:
                continue
            comp_row  = comp_gaps.iloc[0]
            comp_cal  = comp_row["CalvingDate"]
            comp_plac = int(comp_row["PreLac"])
            comp_qlac = int(comp_row["PostLac"])
            comp_pdim = int(comp_row["PreDIM"])
            comp_gs   = pd.Timestamp(comp_row["GapStart"]).date()
            d_ts      = pd.Timestamp(d)
            if not pd.isnull(comp_cal) and d_ts >= comp_cal:
                c_lac = comp_qlac
                c_dim = (d_ts - comp_cal).days
            else:
                c_lac = comp_plac
                c_dim = comp_pdim + (d_ts - pd.Timestamp(comp_gs)).days
            if c_dim <= 0:
                continue
            cp = get_woods_pred(comp_aid, c_lac, c_dim)
            if cp is not None and cp >= 3.0:
                competitor_preds[comp_aid] = cp

        if len(competitor_preds) == 0:
            all_days_ok = False; break

        # ── Step 3: pre-filter pool ──
        pool["_err"] = (pool["TotalMilkYieldSessionKg"] - pred).abs()
        pool_filtered = pool[pool["_err"] <= FILTER_STD * YIELD_STD].copy()
        if len(pool_filtered) == 0:
            all_days_ok = False; break

        # ── Step 4: cost matrix + Hungarian ──
        animal_list = list(competitor_preds.keys())
        n_ani = len(animal_list)
        n_rec = len(pool_filtered)

        pool_filtered = pool_filtered.reset_index(drop=True)

        yields   = pool_filtered["TotalMilkYieldSessionKg"].values
        flows    = pool_filtered["AverageMilkFlowKgPerMin"].values
        durs     = pool_filtered["MilkingDurationSeconds"].values

        yield_std = float(pool_filtered["TotalMilkYieldSessionKg"].std()) or YIELD_STD
        flow_std  = float(pool_filtered["AverageMilkFlowKgPerMin"].std())  or 1.0
        dur_std   = float(pool_filtered["MilkingDurationSeconds"].std())   or 60.0

        cost = np.full((n_ani, n_rec), 999.0)
        for i, a in enumerate(animal_list):
            p = competitor_preds[a]
            yield_errs = np.abs(yields - p) / yield_std
            flow_errs  = np.abs(flows  - np.nanmean(flows)) / flow_std
            dur_errs   = np.abs(durs   - np.nanmean(durs))  / dur_std
            row_costs  = 10 * yield_errs + flow_errs + dur_errs
            cost[i]    = np.nan_to_num(row_costs, nan=999.0, posinf=999.0, neginf=999.0)

        row_ind, col_ind = linear_sum_assignment(cost)

        # find our animal's assignment
        if aid not in animal_list:
            all_days_ok = False; break

        our_row = animal_list.index(aid)
        if our_row not in list(row_ind):
            all_days_ok = False; break

        our_col       = col_ind[list(row_ind).index(our_row)]
        assigned_idx  = int(our_col)  # already reset_index so positional = label
        assigned_yield = yields[assigned_idx]

        # ── Step 5: confidence filter ±0.5 kg ──
        if abs(assigned_yield - pred) > 0.5:
            day_results[d] = {"passed": False, "correct": False,
                               "err": abs(assigned_yield - pred)}
        else:
            # check correctness by yield value instead of index (index shifted after reset)
            injected_yield = held_out[
                held_out["EventDate"].dt.date == d
            ]["TotalMilkYieldSessionKg"].values
            correct = (len(injected_yield) > 0 and
                       abs(assigned_yield - injected_yield[0]) < 0.01)
            day_results[d] = {"passed": True, "correct": correct,
                               "err": abs(assigned_yield - pred),
                               "assigned_idx": assigned_idx}

    # ── clean up injected records ──
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    if not day_results:
        continue

    n_passed  = sum(1 for v in day_results.values() if v["passed"])
    n_correct = sum(1 for v in day_results.values() if v.get("correct", False))
    mean_err  = np.mean([v["err"] for v in day_results.values() if v["passed"]]) if n_passed > 0 else None

    val6_results.append({
        "AnimalId":   aid,
        "days_passed_filter": n_passed,
        "days_correct":       n_correct,
        "days_total":         len(day_results),
        "pct_passed":         n_passed / WINDOW,
        "pct_correct":        n_correct / n_passed if n_passed > 0 else 0,
        "mean_err":           mean_err,
    })

df_v6 = pd.DataFrame(val6_results)
print(f"\nFull 6-step pipeline validation ({len(df_v6)} animals):")
print(f"  Days passing ±0.5kg filter : {df_v6['pct_passed'].mean()*100:.1f}%")
print(f"  Of those — correct record  : {df_v6[df_v6['days_passed_filter']>0]['pct_correct'].mean()*100:.1f}%")
print(f"  Mean yield error (passed)  : {df_v6[df_v6['mean_err'].notna()]['mean_err'].mean():.4f} kg")
print(f"\n  Animals where ALL 7 days passed filter : {(df_v6['days_passed_filter']==WINDOW).sum()}")
print(f"  Animals where ALL 7 days correct       : {(df_v6['days_correct']==WINDOW).sum()}")
print(f"\n  Distribution of correct days:")
print(df_v6['days_correct'].value_counts().sort_index())

Full pipeline validation: 100%|██████████| 200/200 [02:26<00:00,  1.36it/s]


Full 6-step pipeline validation (27 animals):
  Days passing ±0.5kg filter : 42.3%
  Of those — correct record  : 57.3%
  Mean yield error (passed)  : 0.1423 kg

  Animals where ALL 7 days passed filter : 7
  Animals where ALL 7 days correct       : 0

  Distribution of correct days:
days_correct
0    13
1     5
2     5
3     4
Name: count, dtype: int64


In [ ]:
# ── FULL RUN ON MISSING DATA ──
# Run only if validation accuracy is satisfactory (>60% recommended)
# This finds trajectories for unrecovered gap animals in DfMissing

# Load Hungarian results to know which records are already recovered
DfHungarian = pd.read_csv(ProjectRoot / "DfHungarian.csv")
already_recovered = set(DfHungarian.index)  # adjust if index differs

traj_results = []

gap_animals = DfGaps["AnimalId"].astype(str).unique()
all_dates   = sorted(AnimalsByDate.keys())

for aid in tqdm(gap_animals[:500], desc="Trajectory matching"):  # start with 500
    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) == 0:
        continue
    gap_row = gap_rows_a.iloc[0]
    gap_start = gap_row["GapStart"].date()
    gap_end   = gap_row["GapEnd"].date()

    # Get all dates in this gap
    gap_dates = [d for d in all_dates if gap_start <= d <= gap_end]
    if len(gap_dates) < WINDOW:
        continue

    # Slide window across gap dates
    for start_i in range(len(gap_dates) - WINDOW + 1):
        window_dates = gap_dates[start_i : start_i + WINDOW]
        for session in [1, 2, 3]:
            result = score_trajectory(aid, gap_row, window_dates, session)
            if result is not None:
                mean_err, idxs, preds = result
                traj_results.append({
                    "AnimalId": aid,
                    "WindowStart": window_dates[0],
                    "WindowEnd":   window_dates[-1],
                    "Session":     session,
                    "MeanErr":     mean_err,
                    "RecordIdxs":  idxs,
                })
                break  # found match for this window, move to next

DfTraj = pd.DataFrame(traj_results)
print(f"\nTrajectory matches found: {len(DfTraj):,}")
print(f"Unique animals matched  : {DfTraj['AnimalId'].nunique():,}")
DfTraj.to_csv(ProjectRoot / "DfTrajMatches.csv", index=False)
print("Saved.")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy import stats
import numpy as np

# We already have df_v6 from Cell 8 (full pipeline validation)
# Each row is one animal, days_passed_filter and days_correct are known

# Treat each DAY as one prediction instance
# True positive:  day passed filter AND correct record
# False positive: day passed filter AND wrong record
# False negative: day did NOT pass filter (record exists but we didn't recover it)

all_tp = df_v6["days_correct"].sum()
all_fp = (df_v6["days_passed_filter"] - df_v6["days_correct"]).sum()
all_fn = (df_v6["days_total"] - df_v6["days_passed_filter"]).sum()

precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
recall    = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0

# Bootstrap confidence intervals (1000 resamples)
np.random.seed(42)
n = len(df_v6)
boot_p, boot_r, boot_f = [], [], []

for _ in range(1000):
    idx    = np.random.choice(n, n, replace=True)
    sample = df_v6.iloc[idx]
    tp = sample["days_correct"].sum()
    fp = (sample["days_passed_filter"] - sample["days_correct"]).sum()
    fn = (sample["days_total"] - sample["days_passed_filter"]).sum()
    p  = tp/(tp+fp) if (tp+fp)>0 else 0
    r  = tp/(tp+fn) if (tp+fn)>0 else 0
    f  = 2*p*r/(p+r) if (p+r)>0 else 0
    boot_p.append(p); boot_r.append(r); boot_f.append(f)

p_ci = np.percentile(boot_p, [2.5, 97.5])
r_ci = np.percentile(boot_r, [2.5, 97.5])
f_ci = np.percentile(boot_f, [2.5, 97.5])

print("=== Full pipeline validation — Precision / Recall / F1 ===")
print(f"  n animals tested : {n}")
print(f"  Total days       : {df_v6['days_total'].sum()}")
print(f"  TP               : {all_tp}  (passed filter AND correct)")
print(f"  FP               : {all_fp}  (passed filter BUT wrong record)")
print(f"  FN               : {all_fn}  (did not pass filter — not recovered)")
print()
print(f"  Precision : {precision:.4f}  95% CI [{p_ci[0]:.4f}, {p_ci[1]:.4f}]")
print(f"  Recall    : {recall:.4f}  95% CI [{r_ci[0]:.4f}, {r_ci[1]:.4f}]")
print(f"  F1        : {f1:.4f}  95% CI [{f_ci[0]:.4f}, {f_ci[1]:.4f}]")
print()
print("  Interpretation:")
print(f"  When we recover a record, it is correct {precision*100:.1f}% of the time (precision)")
print(f"  Of all records that exist, we recover {recall*100:.1f}% of them (recall)")
print(f"  Combined F1 score: {f1:.3f}")

=== Full pipeline validation — Precision / Recall / F1 ===
  n animals tested : 27
  Total days       : 136
  TP               : 27  (passed filter AND correct)
  FP               : 53  (passed filter BUT wrong record)
  FN               : 56  (did not pass filter — not recovered)

  Precision : 0.3375  95% CI [0.1712, 0.5741]
  Recall    : 0.3253  95% CI [0.2499, 0.4026]
  F1        : 0.3313  95% CI [0.2138, 0.4306]

  Interpretation:
  When we recover a record, it is correct 33.8% of the time (precision)
  Of all records that exist, we recover 32.5% of them (recall)
  Combined F1 score: 0.331


In [ ]:
# Measure how much iterative enrichment actually added
# Load the final output to see how many came from enrichment vs first pass

import os

if os.path.exists(ProjectRoot / "AnimalIDRecovered_Final.csv"):
    DfFinal = pd.read_csv(ProjectRoot / "AnimalIDRecovered_Final.csv", low_memory=False)
    print("=== Iterative enrichment breakdown ===")
    print(DfFinal["RecoveryMethod"].value_counts())
    print()
    print(f"Total rows         : {len(DfFinal):,}")
    print(f"Original known     : {(DfFinal['RecoveryMethod']=='Original').sum():,}")
    print(f"Recovered          : {(DfFinal['RecoveryMethod']=='Recovered').sum():,}")
    print(f"Unrecovered        : {(DfFinal['RecoveryMethod']=='Unrecovered').sum():,}")
else:
    print("AnimalIDRecovered_Final.csv not found — run from the main notebook")
    print()
    # Instead, report what we know from the methodology:
    print("From the Hungarian run:")
    print("  First pass (DIM<305 filter)    : 521,414 records")
    print("  After enrichment (final)       : 586,311 records")
    print("  Added by iterative enrichment  : 64,897 records (+12.5%)")
    print("  Mean error improvement         : from 0.135kg to 0.124kg")

=== Iterative enrichment breakdown ===
RecoveryMethod
Original       6737364
Unrecovered    1114692
Recovered       586311
Name: count, dtype: int64

Total rows         : 8,438,367
Original known     : 6,737,364
Recovered          : 586,311
Unrecovered        : 1,114,692


In [ ]:
np.random.seed(42)
sample_unfiltered = np.random.choice(candidates_val, min(200, len(candidates_val)), replace=False)

unfiltered_results = []

for aid in tqdm(sample_unfiltered, desc="Unfiltered validation"):
    aid = str(aid)
    animal_recs  = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 10:
        continue

    session      = int(animal_recs["MilkingSession"].value_counts().idxmax())
    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    mask     = (animal_recs["EventDate"].dt.date.isin(window_dates) &
                (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_recs  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]]
        post_recs = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]]
        if len(pre_recs) == 0 or len(post_recs) == 0:
            continue
        pre_rec  = pre_recs.iloc[-1]
        post_rec = post_recs.iloc[0]
        gap_row  = pd.Series({
            "AnimalId": aid,
            "GapStart": pd.Timestamp(window_dates[0]),
            "GapEnd":   pd.Timestamp(window_dates[-1]),
            "PreDIM":   pre_rec.get("DaysInMilk", 150),
            "PreLac":   pre_rec.get("LactationNumber", 1),
            "PostLac":  post_rec.get("LactationNumber", 1),
            "CalvingDate": pd.NaT,
        })

    calving   = gap_row["CalvingDate"]
    pre_lac   = int(gap_row["PreLac"])
    post_lac  = int(gap_row["PostLac"])
    pre_dim   = int(gap_row["PreDIM"])
    gap_start = pd.Timestamp(gap_row["GapStart"]).date()

    # get Wood's predictions
    day_preds = {}
    valid = True
    for d in window_dates:
        d_ts = pd.Timestamp(d)
        lac  = post_lac if (not pd.isnull(calving) and d_ts >= calving) else pre_lac
        dim  = ((d_ts - calving).days if (not pd.isnull(calving) and d_ts >= calving)
                else pre_dim + (d_ts - pd.Timestamp(gap_start)).days)
        if dim <= 0: valid = False; break
        pred = get_woods_pred(aid, lac, dim)
        if pred is None or pred < 3.0: valid = False; break
        day_preds[d] = pred
    if not valid:
        continue

    # inject held-out records
    injected_keys = []; injected_yields = {}
    for d in window_dates:
        key     = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0: continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(columns=Features+["EventDate","MilkingSession"])
        MysteryByDateSession[key] = pd.concat([MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)
        injected_yields[d] = float(row["TotalMilkYieldSessionKg"].values[0])

    # run per-day Hungarian — NO confidence filter
    day_results = {}
    for d in window_dates:
        pred = day_preds[d]
        key  = (d, session)
        if key not in MysteryByDateSession or len(MysteryByDateSession[key]) == 0:
            break

        pool = MysteryByDateSession[key].copy().reset_index(drop=True)
        absent_aids = [str(x) for x in AnimalsByDate.get(d, [])]
        if aid not in absent_aids: absent_aids.append(aid)

        competitor_preds = {}
        for comp_aid in absent_aids:
            cg = DfGaps[DfGaps["AnimalId"].astype(str) == comp_aid]
            if len(cg) == 0: continue
            cr    = cg.iloc[0]
            cc    = cr["CalvingDate"]
            d_ts  = pd.Timestamp(d)
            c_lac = int(cr["PostLac"]) if (not pd.isnull(cc) and d_ts >= cc) else int(cr["PreLac"])
            c_dim = ((d_ts - cc).days if (not pd.isnull(cc) and d_ts >= cc)
                     else int(cr["PreDIM"]) + (d_ts - pd.Timestamp(cr["GapStart"].date())).days)
            if c_dim <= 0: continue
            cp = get_woods_pred(comp_aid, c_lac, c_dim)
            if cp and cp >= 3.0: competitor_preds[comp_aid] = cp

        if len(competitor_preds) == 0: break

        animal_list = list(competitor_preds.keys())
        n_ani = len(animal_list)
        n_rec = len(pool)

        yields    = pool["TotalMilkYieldSessionKg"].values
        flows     = pool["AverageMilkFlowKgPerMin"].values
        durs      = pool["MilkingDurationSeconds"].values
        y_std     = float(pool["TotalMilkYieldSessionKg"].std()) or YIELD_STD
        fl_std    = float(pool["AverageMilkFlowKgPerMin"].std()) or 1.0
        d_std     = float(pool["MilkingDurationSeconds"].std())  or 60.0

        cost = np.full((n_ani, n_rec), 999.0)
        for i, a in enumerate(animal_list):
            p         = competitor_preds[a]
            row_costs = (10*np.abs(yields-p)/y_std +
                         np.abs(flows-np.nanmean(flows))/fl_std +
                         np.abs(durs-np.nanmean(durs))/d_std)
            cost[i]   = np.nan_to_num(row_costs, nan=999.0, posinf=999.0, neginf=999.0)

        row_ind, col_ind = linear_sum_assignment(cost)
        if aid not in animal_list: break
        our_row = animal_list.index(aid)
        if our_row not in list(row_ind): break

        our_col        = col_ind[list(row_ind).index(our_row)]
        assigned_yield = float(yields[our_col])
        true_y         = injected_yields.get(d)
        correct        = (true_y is not None and abs(assigned_yield - true_y) < 0.01)
        err            = abs(assigned_yield - pred)

        day_results[d] = {
            "correct":        correct,
            "err":            err,
            "passed_filter":  err <= 0.5,   # record whether it WOULD have passed
        }

    # clean up
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    if not day_results: continue

    n_total_days    = len(day_results)
    n_correct_all   = sum(1 for v in day_results.values() if v["correct"])
    n_correct_pass  = sum(1 for v in day_results.values() if v["correct"] and v["passed_filter"])
    n_pass          = sum(1 for v in day_results.values() if v["passed_filter"])
    mean_err_all    = np.mean([v["err"] for v in day_results.values()])
    mean_err_pass   = np.mean([v["err"] for v in day_results.values() if v["passed_filter"]]) if n_pass > 0 else None

    unfiltered_results.append({
        "AnimalId":          aid,
        "days_total":        n_total_days,
        "correct_all":       n_correct_all,
        "correct_pass":      n_correct_pass,
        "days_pass_filter":  n_pass,
        "mean_err_all":      mean_err_all,
        "mean_err_pass":     mean_err_pass,
    })

df_uf = pd.DataFrame(unfiltered_results)
n = len(df_uf)
total_days   = df_uf["days_total"].sum()
total_correct_all  = df_uf["correct_all"].sum()
total_correct_pass = df_uf["correct_pass"].sum()
total_pass   = df_uf["days_pass_filter"].sum()

print(f"Unfiltered Hungarian validation ({n} animals, {total_days} days total)")
print()
print(f"  ── WITHOUT confidence filter ──")
print(f"  Correct assignments  : {total_correct_all} / {total_days}")
print(f"  Precision (no filter): {total_correct_all/total_days:.4f}  ({100*total_correct_all/total_days:.1f}%)")
print(f"  Mean yield error     : {df_uf['mean_err_all'].mean():.4f} kg")
print()
print(f"  ── WITH ±0.5 kg confidence filter ──")
print(f"  Days passing filter  : {total_pass} / {total_days}  ({100*total_pass/total_days:.1f}%)")
print(f"  Correct when passed  : {total_correct_pass} / {total_pass}")
print(f"  Precision (filtered) : {total_correct_pass/total_pass:.4f}  ({100*total_correct_pass/total_pass:.1f}%)") if total_pass > 0 else None
print(f"  Mean yield error     : {df_uf['mean_err_pass'].mean():.4f} kg")
print()
print(f"  ── FILTER EFFECT ──")
print(f"  Precision gain from filter : +{100*(total_correct_pass/total_pass - total_correct_all/total_days):.1f} percentage points")
print(f"  Coverage cost of filter    : -{100*(1-total_pass/total_days):.1f}% of days not assigned")

Unfiltered validation: 100%|██████████| 200/200 [02:28<00:00,  1.35it/s]

Unfiltered Hungarian validation (36 animals, 252 days total)

  ── WITHOUT confidence filter ──
  Correct assignments  : 245 / 252
  Precision (no filter): 0.9722  (97.2%)
  Mean yield error     : 1.9068 kg

  ── WITH ±0.5 kg confidence filter ──
  Days passing filter  : 76 / 252  (30.2%)
  Correct when passed  : 69 / 76
  Precision (filtered) : 0.9079  (90.8%)
  Mean yield error     : 0.2139 kg

  ── FILTER EFFECT ──
  Precision gain from filter : +-6.4 percentage points
  Coverage cost of filter    : -69.8% of days not assigned


In [ ]:
from scipy.optimize import curve_fit, linear_sum_assignment
import datetime

def WoodsCurve(T, A, B, C):
    return A * (T**B) * np.exp(-C * T)

def fit_woods_oos(aid, lac, exclude_dates):
    """Fit Wood's curve excluding the held-out window dates."""
    recs = DfKnown[
        (DfKnown["AnimalId"] == str(aid)) &
        (DfKnown["LactationNumber"] == int(lac)) &
        (~DfKnown["EventDate"].dt.date.isin(set(exclude_dates))) &
        (DfKnown["DaysInMilk"] > 0) &
        (DfKnown["TotalMilkYieldSessionKg"] > 3.0)
    ]
    if len(recs) < 10:
        return None
    t = recs["DaysInMilk"].values.astype(float)
    y = recs["TotalMilkYieldSessionKg"].values.astype(float)
    try:
        popt, _ = curve_fit(WoodsCurve, t, y,
                            p0=[20, 0.2, 0.003],
                            bounds=([0,0,0],[200,2,0.1]),
                            maxfev=5000)
        return tuple(popt)
    except:
        return None

print("OOS Wood's fitter ready")

OOS Wood's fitter ready


In [ ]:
np.random.seed(42)
sample_oos = np.random.choice(candidates_val, min(200, len(candidates_val)), replace=False)

oos_results = []

for aid in tqdm(sample_oos, desc="OOS validation"):
    aid = str(aid)
    animal_recs  = DfKnown[DfKnown["AnimalId"] == aid].sort_values("EventDate")
    dates_sorted = sorted(animal_recs["EventDate"].dt.date.unique())
    if len(dates_sorted) < WINDOW + 10:
        continue

    session      = int(animal_recs["MilkingSession"].value_counts().idxmax())
    start_idx    = np.random.randint(5, len(dates_sorted) - WINDOW - 2)
    window_dates = dates_sorted[start_idx : start_idx + WINDOW]

    mask     = (animal_recs["EventDate"].dt.date.isin(window_dates) &
                (animal_recs["MilkingSession"] == session))
    held_out = animal_recs[mask]
    if len(held_out) < WINDOW:
        continue

    gap_rows_a = DfGaps[DfGaps["AnimalId"].astype(str) == aid]
    if len(gap_rows_a) > 0:
        gap_row = gap_rows_a.iloc[0]
    else:
        pre_recs  = animal_recs[animal_recs["EventDate"].dt.date < window_dates[0]]
        post_recs = animal_recs[animal_recs["EventDate"].dt.date > window_dates[-1]]
        if len(pre_recs) == 0 or len(post_recs) == 0:
            continue
        gap_row = pd.Series({
            "AnimalId": aid,
            "GapStart": pd.Timestamp(window_dates[0]),
            "GapEnd":   pd.Timestamp(window_dates[-1]),
            "PreDIM":   int(pre_recs.iloc[-1].get("DaysInMilk", 150)),
            "PreLac":   int(pre_recs.iloc[-1].get("LactationNumber", 1)),
            "PostLac":  int(post_recs.iloc[0].get("LactationNumber", 1)),
            "CalvingDate": pd.NaT,
        })

    calving   = gap_row["CalvingDate"]
    pre_lac   = int(gap_row["PreLac"])
    post_lac  = int(gap_row["PostLac"])
    pre_dim   = int(gap_row["PreDIM"])
    gap_start = pd.Timestamp(gap_row["GapStart"]).date()

    # ── refit Wood's curve EXCLUDING the held-out window ──
    lac_to_fit = post_lac if (not pd.isnull(calving)) else pre_lac
    oos_params = fit_woods_oos(aid, lac_to_fit, window_dates)
    if oos_params is None:
        # fallback: try pre_lac
        oos_params = fit_woods_oos(aid, pre_lac, window_dates)
    if oos_params is None:
        continue
    A_oos, B_oos, C_oos = oos_params

    # get per-day predictions using OOS curve
    day_preds = {}
    valid = True
    for d in window_dates:
        d_ts = pd.Timestamp(d)
        if not pd.isnull(calving) and d_ts >= calving:
            dim = (d_ts - calving).days
        else:
            dim = pre_dim + (d_ts - pd.Timestamp(gap_start)).days
        if dim <= 0:
            valid = False; break
        try:
            pred = A_oos * (dim**B_oos) * np.exp(-C_oos * dim)
        except:
            valid = False; break
        if pred < 3.0 or not np.isfinite(pred):
            valid = False; break
        day_preds[d] = float(pred)
    if not valid:
        continue

    # inject held-out records
    injected_keys = []; injected_yields = {}
    for d in window_dates:
        key     = (d, session)
        day_rec = held_out[held_out["EventDate"].dt.date == d]
        if len(day_rec) == 0: continue
        row = day_rec[Features + ["EventDate","MilkingSession"]].iloc[[0]].copy()
        if key not in MysteryByDateSession:
            MysteryByDateSession[key] = pd.DataFrame(columns=Features+["EventDate","MilkingSession"])
        MysteryByDateSession[key] = pd.concat([MysteryByDateSession[key], row], ignore_index=True)
        injected_keys.append(key)
        injected_yields[d] = float(row["TotalMilkYieldSessionKg"].values[0])

    # per-day Hungarian (same as unfiltered validation)
    day_results = {}
    for d in window_dates:
        pred = day_preds[d]
        key  = (d, session)
        if key not in MysteryByDateSession or len(MysteryByDateSession[key]) == 0:
            break

        pool = MysteryByDateSession[key].copy().reset_index(drop=True)

        absent_aids = [str(x) for x in AnimalsByDate.get(d, [])]
        if aid not in absent_aids: absent_aids.append(aid)

        competitor_preds = {}
        for comp_aid in absent_aids:
            cg = DfGaps[DfGaps["AnimalId"].astype(str) == comp_aid]
            if len(cg) == 0: continue
            cr   = cg.iloc[0]; cc = cr["CalvingDate"]
            d_ts = pd.Timestamp(d)
            c_lac = int(cr["PostLac"]) if (not pd.isnull(cc) and d_ts >= cc) else int(cr["PreLac"])
            c_dim = ((d_ts-cc).days if (not pd.isnull(cc) and d_ts >= cc)
                     else int(cr["PreDIM"])+(d_ts-pd.Timestamp(cr["GapStart"].date())).days)
            if c_dim <= 0: continue
            cp = get_woods_pred(comp_aid, c_lac, c_dim)
            if cp and cp >= 3.0: competitor_preds[comp_aid] = cp

        if len(competitor_preds) == 0: break
        # use OOS pred for our animal
        competitor_preds[aid] = pred

        animal_list = list(competitor_preds.keys())
        n_ani = len(animal_list); n_rec = len(pool)
        yields = pool["TotalMilkYieldSessionKg"].values
        flows  = pool["AverageMilkFlowKgPerMin"].values
        durs   = pool["MilkingDurationSeconds"].values
        y_std  = float(pool["TotalMilkYieldSessionKg"].std()) or YIELD_STD
        fl_std = float(pool["AverageMilkFlowKgPerMin"].std())  or 1.0
        d_std  = float(pool["MilkingDurationSeconds"].std())   or 60.0

        cost = np.full((n_ani, n_rec), 999.0)
        for i, a in enumerate(animal_list):
            p = competitor_preds[a]
            row_c = (10*np.abs(yields-p)/y_std +
                     np.abs(flows-np.nanmean(flows))/fl_std +
                     np.abs(durs-np.nanmean(durs))/d_std)
            cost[i] = np.nan_to_num(row_c, nan=999.0, posinf=999.0, neginf=999.0)

        row_ind, col_ind = linear_sum_assignment(cost)
        if aid not in animal_list: break
        our_row = animal_list.index(aid)
        if our_row not in list(row_ind): break
        our_col        = col_ind[list(row_ind).index(our_row)]
        assigned_yield = float(yields[our_col])
        true_y         = injected_yields.get(d)
        correct        = (true_y is not None and abs(assigned_yield - true_y) < 0.01)
        err            = abs(assigned_yield - pred)

        day_results[d] = {
            "correct":       correct,
            "err":           err,
            "passed_filter": err <= 0.5,
        }

    # clean up
    for key in injected_keys:
        df_pool = MysteryByDateSession[key]
        MysteryByDateSession[key] = df_pool[
            ~df_pool["EventDate"].dt.date.isin(window_dates)
        ].reset_index(drop=True)

    if not day_results: continue

    n_days         = len(day_results)
    n_correct_all  = sum(1 for v in day_results.values() if v["correct"])
    n_correct_pass = sum(1 for v in day_results.values() if v["correct"] and v["passed_filter"])
    n_pass         = sum(1 for v in day_results.values() if v["passed_filter"])

    oos_results.append({
        "AnimalId":         aid,
        "days_total":       n_days,
        "correct_all":      n_correct_all,
        "correct_pass":     n_correct_pass,
        "days_pass_filter": n_pass,
    })

df_oos = pd.DataFrame(oos_results)
n = len(df_oos)
td   = df_oos["days_total"].sum()
ca   = df_oos["correct_all"].sum()
cp   = df_oos["correct_pass"].sum()
tp   = df_oos["days_pass_filter"].sum()

print(f"OOS Wood's validation ({n} animals, {td} days)")
print()
print(f"  WITHOUT filter: {ca}/{td} = {100*ca/td:.1f}% precision")
print(f"  WITH filter   : {cp}/{tp} = {100*cp/tp:.1f}% precision  ({tp}/{td} = {100*tp/td:.1f}% pass rate)")
print()
print(f"  Compare to leaky version:")
print(f"    WITHOUT filter: 97.2%")
print(f"    WITH filter   : 90.8%")
print()
print(f"  Leakage effect: {97.2 - 100*ca/td:+.1f}pp (no filter)  /  {90.8 - 100*cp/tp if tp>0 else 'n/a':+.1f}pp (with filter)")

OOS validation: 100%|██████████| 200/200 [06:36<00:00,  1.98s/it]

OOS Wood's validation (36 animals, 252 days)

  WITHOUT filter: 245/252 = 97.2% precision
  WITH filter   : 43/50 = 86.0% precision  (50/252 = 19.8% pass rate)

  Compare to leaky version:
    WITHOUT filter: 97.2%
    WITH filter   : 90.8%

  Leakage effect: -0.0pp (no filter)  /  +4.8pp (with filter)
